In [144]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [145]:
load_dotenv()
API_KEY = os.getenv("CENSUS_API_KEY")

In [146]:
variables = ",".join([
    "NAME",            # County name
    "DP03_0062E",      # Median household income (in 2024 inflation-adjusted dollars).
    "DP04_0091E",      # Number of Owners with a mortgage.
    "DP04_0101E",      # Monthly Owner Costs (with a mortgage) Median (dollars).
    "DP02_0053PE",     # Percent of population 3+ enrolled in school.
    "DP02_0057PE",     # Percent of population 3+ Highschool enrolled in school.    
])

In [154]:
employment_variables = ",".join([
    "S2301_C01_002E",  # Population ages 16–19
    "S2301_C03_002E"   # Employment/population ratio ages 16–19
])

In [ ]:
# years = range(2018, 2025)

# all_data = []

# for year in years:

#     print(f"Requesting {year}...")

#     url = f"https://api.census.gov/data/{year}/acs/acs5/profile"

#     params = {
#         "get": variables,
#         "for": "county:*",
#         "in": "state:37",
#         "key": API_KEY
#     }

#     try:
#         response = requests.get(url, params=params, timeout=10)

#         print(f"Status code: {response.status_code}")

#         if response.status_code == 200:

#             data = response.json()

#             df_year = pd.DataFrame(
#                 data[1:],
#                 columns=data[0]
#             )

#             df_year["year"] = year

#             all_data.append(df_year)

#             print(f"{year} complete!")

#         else:
#             print(f"{year}: Error {response.status_code}")
#             print(response.text)

#     except requests.exceptions.Timeout:
#         print(f"{year}: Request timed out")

#     except requests.exceptions.RequestException as e:
#         print(f"{year}: Request failed - {e}")

Requesting 2018...
2018: Request timed out
Requesting 2019...
Status code: 200
2019 complete!
Requesting 2020...
Status code: 200
2020 complete!
Requesting 2021...
Status code: 200
2021 complete!
Requesting 2022...
Status code: 200
2022 complete!
Requesting 2023...
Status code: 200
2023 complete!
Requesting 2024...
Status code: 200
2024 complete!


In [155]:
years = range(2018, 2025)
all_data = []

for year in years:
    print(f"Requesting {year}...")

    # -------------------------
    # PROFILE DATA
    # -------------------------
    profile_url = f"https://api.census.gov/data/{year}/acs/acs5/profile"

    profile_params = {
        "get": variables,
        "for": "county:*",
        "in": "state:37",
        "key": API_KEY
    }

    # -------------------------
    # EMPLOYMENT DATA
    # -------------------------
    employment_url = f"https://api.census.gov/data/{year}/acs/acs5/subject"

    employment_params = {
        "get": employment_variables,
        "for": "county:*",
        "in": "state:37",
        "key": API_KEY
    }

    try:
        # Get profile data
        profile_response = requests.get(
            profile_url,
            params=profile_params,
            timeout=10
        )

        # Get employment data
        employment_response = requests.get(
            employment_url,
            params=employment_params,
            timeout=10
        )

        print(
            f"Profile: {profile_response.status_code} | "
            f"Employment: {employment_response.status_code}"
        )

        if profile_response.status_code == 200 and employment_response.status_code == 200:

            # Convert profile response to dataframe
            profile_data = profile_response.json()

            profile_df = pd.DataFrame(
                profile_data[1:],
                columns=profile_data[0]
            )

            # Convert employment response to dataframe
            employment_data = employment_response.json()

            employment_df = pd.DataFrame(
                employment_data[1:],
                columns=employment_data[0]
            )

            # Merge using state + county
            df_year = profile_df.merge(
                employment_df,
                on=["state", "county"],
                how="left"
            )

            # Add year
            df_year["year"] = year

            all_data.append(df_year)

            print(f"{year} complete!")

        else:
            print(f"{year}: Error")

    except requests.exceptions.Timeout:
        print(f"{year}: Request timed out")

    except requests.exceptions.RequestException as e:
        print(f"{year}: Request failed - {e}")

Requesting 2018...
Profile: 200 | Employment: 200
2018 complete!
Requesting 2019...
Profile: 200 | Employment: 200
2019 complete!
Requesting 2020...
Profile: 200 | Employment: 200
2020 complete!
Requesting 2021...
Profile: 200 | Employment: 200
2021 complete!
Requesting 2022...
Profile: 200 | Employment: 200
2022 complete!
Requesting 2023...
Profile: 200 | Employment: 200
2023 complete!
Requesting 2024...
Profile: 200 | Employment: 200
2024 complete!


In [156]:
df = pd.concat(all_data, ignore_index=True)
df.head()

,NAME,DP03_0062E,DP04_0091E,DP04_0101E,DP02_0053PE,DP02_0057PE,state,county,S2301_C01_002E,S2301_C03_002E,year
0,"Alleghany County, North Carolina",37558,1616,997,5.5,17.5,37,005,584,28.9,2018
1,"Caldwell County, North Carolina",42072,12714,979,5.8,18.5,37,027,3713,30.2,2018
2,"Catawba County, North Carolina",51157,26110,1076,6.0,22.1,37,035,8301,33.5,2018
3,"Forsyth County, North Carolina",50128,60001,1208,5.4,28.5,37,067,21669,25.1,2018
4,"Bladen County, North Carolina",32378,3996,1024,4.9,21.1,37,017,1720,28.3,2018


In [157]:
numeric_cols = [
    #"NAME",            # County name
    "DP03_0062E",      # Income_Median
    "DP04_0091E",      # Owners_with_Mortgage
    "DP04_0101E",      # Median_Owner_Costs
    "DP02_0053PE",     # Enrolled_Percent
    "DP02_0057PE",    # Highschool_Enrolled_Percent
    "S2301_C01_002E",  # Population ages 16–19
    "S2301_C03_002E"   # Employment/population ratio ages 16–19
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce") # coerce -> If something cannot be converted to a number, don’t crash. Instead, turn it into NaN.

df = df.rename(columns={
    #"NAME": "County_Name",
    "DP03_0062E": "Income_Median",
    "DP04_0091E": "Owners_with_Mortgage",
    "DP04_0101E": "Median_Owner_Costs",
    "DP02_0053PE": "Enrolled_Percent",
    "DP02_0057PE": "Highschool_Enrolled_Percent",
    "S2301_C01_002E": "Population_16_19",
    "S2301_C03_002E": "Employment_Ratio_16_19"
})

df.head()

,NAME,Income_Median,Owners_with_Mortgage,Median_Owner_Costs,Enrolled_Percent,Highschool_Enrolled_Percent,state,county,Population_16_19,Employment_Ratio_16_19,year
0,"Alleghany County, North Carolina",37558,1616,997,5.5,17.5,37,005,584,28.9,2018
1,"Caldwell County, North Carolina",42072,12714,979,5.8,18.5,37,027,3713,30.2,2018
2,"Catawba County, North Carolina",51157,26110,1076,6.0,22.1,37,035,8301,33.5,2018
3,"Forsyth County, North Carolina",50128,60001,1208,5.4,28.5,37,067,21669,25.1,2018
4,"Bladen County, North Carolina",32378,3996,1024,4.9,21.1,37,017,1720,28.3,2018


In [158]:
df.isnull().sum()

NAME                           0
Income_Median                  0
Owners_with_Mortgage           0
Median_Owner_Costs             0
Enrolled_Percent               0
Highschool_Enrolled_Percent    0
state                          0
county                         0
Population_16_19               0
Employment_Ratio_16_19         0
year                           0
dtype: int64

In [159]:
print(df.columns.tolist())

['NAME', 'Income_Median', 'Owners_with_Mortgage', 'Median_Owner_Costs', 'Enrolled_Percent', 'Highschool_Enrolled_Percent', 'state', 'county', 'Population_16_19', 'Employment_Ratio_16_19', 'year']


In [162]:
yearly_avg = df.groupby("year")[[
    "Income_Median",
    "Owners_with_Mortgage",
    "Median_Owner_Costs",
    "Enrolled_Percent",
    "Highschool_Enrolled_Percent",
    "Population_16_19",
    "Employment_Ratio_16_19"
]].mean()

yearly_avg

,Income_Median,Owners_with_Mortgage,Median_Owner_Costs,Enrolled_Percent,Highschool_Enrolled_Percent,Population_16_19,Employment_Ratio_16_19
year,,,,,,,
2018,46391.68,16172.13,1179.97,5.109,23.762,5428.05,29.917
2019,48419.49,16336.99,1196.55,25189.070,22.841,5520.01,31.036
2020,50062.75,16562.20,1207.13,25216.150,22.709,5575.21,31.127
2021,53090.87,16611.83,1249.68,25158.650,22.758,5651.68,32.423
2022,58261.46,16904.66,1345.87,25025.640,22.677,5680.06,33.921
2023,61072.21,17217.52,1394.40,25202.850,22.761,5667.36,34.001
2024,63181.72,17487.90,1451.38,25301.020,22.749,5738.57,34.741
